# Daily LightGBM with scarce mule labels on Kaggle CPU

Label-scarcity runs of the research plan (section 19): the daily LightGBM of `python -m earlymule.models.lightgbm_export` (full feature set) trained with the mule labels of only a share of the training cases (`--label-fraction`), chosen by seed from the case map `label_cases_80K-earlymule-mule.csv`. The other training mules count as negatives (`unlabeled = 'negative'`) or leave training (`'drop'`); validation and test labels are untouched. The full-label run is `lgbm_abl_full`, with the same negative sample per seed. CPU only, so it does not use the weekly GPU quota. Each run writes scores and `run_meta.json` to `/kaggle/working/lgbm_lf<F>[_drop]/seed<N>/` as soon as it ends; evaluation runs locally with `python -m earlymule.evaluation.import_scores`.

Inputs: the private datasets `verzol/earlymule-80k-earlymule-mule` (export) and `verzol/earlymule-80k-label-cases` (case map, `python -m earlymule.data.label_fraction`). One notebook per batch is generated by `notebooks/kaggle/label_fraction/make_batches.py`, which rewrites the parameter cell.

In [ ]:
# Run parameters: dataset, (label fraction, unlabeled mode, seed) jobs (fraction 1 = full labels), run one after another; commit to check out
DATASET = '80K-earlymule-mule'
JOBS = [(0.01, 'negative', 0)]
COMMIT = None

In [ ]:
# Clone the repository at COMMIT and install it without dependencies (LightGBM, pandas and PyYAML are preinstalled)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!nproc; free -g

In [ ]:
# Train every job in order; a failed run is logged and the next one still starts
import json, subprocess, time
from pathlib import Path
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if json.loads(m.read_text())['dataset'] == DATASET)
label_cases = next(Path('/kaggle/input').rglob(f'label_cases_{DATASET}.csv'), None)  # only fractions below 1 need it
codes = []
for fraction, unlabeled, seed in JOBS:
    tag = 'lgbm_full' if fraction >= 1 else f'lgbm_lf{fraction:g}' + ('_drop' if unlabeled == 'drop' else '')
    out = Path('/kaggle/working') / tag / f'seed{seed}'
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    cmd = ['python', '-m', 'earlymule.models.lightgbm_export', '--data', str(data), '--ablation', 'full',
           '--seed', str(seed), '--out', str(out)]
    if fraction < 1:
        cmd += ['--label-fraction', str(fraction), '--label-cases', str(label_cases), '--unlabeled', unlabeled]
    code = subprocess.call(cmd, cwd='/tmp/earlymule-tg', stdout=open(out / 'run.log', 'w'), stderr=subprocess.STDOUT)
    codes.append((fraction, unlabeled, seed, code, round((time.time() - t0) / 60, 1)))
    print(codes[-1], flush=True)
print(codes)

In [ ]:
# Last lines of every run log
for log in sorted(Path('/kaggle/working').glob('lgbm_*/seed*/run.log')):
    print(f'--- {log.parent.parent.name}/{log.parent.name}')
    print(log.read_text()[-1500:])